# 单一入口：plan、apply、check，然后才 actor

不要在浏览器里追加 Python。Run All 只调用已随包上传的 `scripts/run_actor_session.py`。
入口会查找 `/kaggle/input/datasets/<owner>/<dataset>/<package>`，也会查找直接挂在 input 或 working 下的旧布局。找到多个准入包就停止，并要求设置 `H3_T4_PACKAGE_ROOT`。
check 超时固定 75 秒。命令阶段预算 240 秒。actor 只在两张 GPU 且剩余时间够 125 秒时启动。
源码保持在只读 input 上。Ray 的 working_dir、会话日志和打包文件写到 `/tmp/h3-ray-runtime`，或写到环境变量 `H3_T4_RUNTIME_DIR`。
结束时单元会打印最终的 `h3-actor-session-manifest.json`。zip 里面的 `session-status.json` 是打包前快照，`bundle_status` 为 `NOT_SEALED`；最终哈希以 zip 旁边的 manifest 和 `.sha256` 为准。缺文件会打印 MISSING。
No persistence：新 GPU 会话从头 bootstrap。不下载权重。


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ENTRY_SCRIPT = Path("scripts") / "run_actor_session.py"
MARKER_SCRIPT = Path("scripts") / "bootstrap_actor_admission.py"
MAX_DEPTH = 6


def default_seeds() -> list[Path]:
    return [Path.cwd(), Path("/kaggle/input"), Path("/kaggle/working")]


def find_package_root(seeds: list[Path] | None = None, environ: dict[str, str] | None = None) -> Path:
    """Return the only directory that contains the current admission entry.

    ``H3_T4_PACKAGE_ROOT`` selects one directory and disables the search.
    A modern Kaggle mount is ``/kaggle/input/datasets/<owner>/<dataset>/<package>``.
    Older sessions mounted the package directly under ``/kaggle/input`` or
    ``/kaggle/working``. More than one match is an error.
    """

    env = os.environ if environ is None else environ
    pinned = env.get("H3_T4_PACKAGE_ROOT", "").strip()
    if pinned:
        root = Path(pinned)
        if not _is_admission_package(root):
            raise SystemExit(f"H3_T4_PACKAGE_ROOT is not the admission package: {root}")
        return root.resolve()
    found: list[Path] = []
    seen: set[str] = set()
    for seed in seeds if seeds is not None else default_seeds():
        if not seed.exists():
            continue
        for match in _scan(seed.resolve(), MAX_DEPTH):
            key = str(match.resolve())
            if key not in seen:
                seen.add(key)
                found.append(match.resolve())
    if not found:
        raise SystemExit(
            "admission package not found. Upload the folder that contains "
            "scripts/run_actor_session.py or set H3_T4_PACKAGE_ROOT."
        )
    if len(found) > 1:
        lines = "\n".join(str(path) for path in found)
        raise SystemExit("multiple admission packages found; set H3_T4_PACKAGE_ROOT to one:\n" + lines)
    return found[0]


def _is_admission_package(root: Path) -> bool:
    return (root / ENTRY_SCRIPT).is_file() and (root / MARKER_SCRIPT).is_file()


def _scan(seed: Path, max_depth: int) -> list[Path]:
    matches: list[Path] = []
    for current, dirnames, _filenames in os.walk(seed):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(seed).parts)
        except ValueError:
            depth = max_depth + 1
        if depth > max_depth:
            dirnames.clear()
            continue
        if _is_admission_package(current_path):
            matches.append(current_path)
            dirnames.clear()
    return matches

root = find_package_root()
evidence = Path("/kaggle/working/h3-actor-session")
code = subprocess.call(
    [sys.executable, str(root / "scripts" / "run_actor_session.py"), "--evidence-dir", str(evidence), "--budget-s", "240"],
    cwd=str(root),
)
spec = importlib.util.spec_from_file_location("h3_actor_session", root / "scripts" / "run_actor_session.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
module.show_session_evidence(evidence)
print("process_exit", code)
